# Cross-Session Comparison — Cheese Acidification

**Sessions:** 28 April 2026 & 5 May 2026  
**Authors:** Lilandra Albert-Lavault & Samuele Moungang  
**Supervisor:** Dimitri Bocquel

This notebook merges both experimental sessions to:
1. Compare corrected pH profiles (aligned to t = 0 = moulage)
2. Compare logistic decline parameters (L, k, t₀) and correlate with fresh yield
3. Compare Luedeking-Piret ODE parameters (μ, α, β) and correlate with fresh yield


---
## 0. Setup & Constants

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import curve_fit, minimize
from scipy.integrate import solve_ivp
from scipy.signal import savgol_filter

plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 120,
                     'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Session metadata ──────────────────────────────────────────────────────
SESSIONS = {
    'S1': {
        'label':        'Session 1 — 28 Apr 2026',
        'color':        'seagreen',
        'yield_pct':    12.17,   # fresh yield from fiche
        'T0_moulage':   pd.Timestamp('2026-04-28 11:36:00'),
        'FW_START':     pd.Timestamp('2026-04-28 12:10:00'),
        'FW_END':       pd.Timestamp('2026-04-28 14:00:00'),
        'cultures':     'MFR38 (974g) + MFR32 (282g)  [ratio 3.45:1]',
    },
    'S2': {
        'label':        'Session 2 — 5 May 2026',
        'color':        'steelblue',
        'yield_pct':    11.80,
        'T0_moulage':   pd.Timestamp('2026-05-05 11:30:00'),  # estimated
        'FW_START':     pd.Timestamp('2026-05-05 11:30:00'),
        'FW_END':       pd.Timestamp('2026-05-05 15:15:00'),
        'cultures':     'MFR38 + MFR32 [ratio 1:2]',
    },
}

T_CAL = 25.0
PH_ISO = 7.0

def nernst_correct(pH_meas, T_meas_C, T_cal_C=T_CAL, pH_iso=PH_ISO):
    T_meas_K = T_meas_C + 273.15
    T_cal_K  = T_cal_C  + 273.15
    return pH_iso + (pH_meas - pH_iso) * (T_meas_K / T_cal_K)

print('Setup OK — Sessions:', list(SESSIONS.keys()))


---
## 1. Load & Preprocess Both Sessions

Reference probe: **Hannah** (has internal ATC → no Nernst correction needed).
USB and Server: Nernst correction applied with assumed T = 35 °C.


In [ ]:
# ── Session 1 — Hannah (PHLOT001) ───────────────────────────────────────
h1_raw = pd.read_csv(
    'data/trace pH fromage 3_28042026_hannah.csv',
    skiprows=20, header=None,
    names=['_','rec','date','time','pH','u1','mV','u2','temp_C','u3','_2'],
    usecols=['date','time','pH','temp_C'], encoding='latin-1',
)
h1_raw['datetime'] = pd.to_datetime(
    h1_raw['date'].str.strip() + ' ' + h1_raw['time'].str.strip(), dayfirst=False)
h1_raw['pH'] = pd.to_numeric(h1_raw['pH'], errors='coerce')
h1_raw['temp_C'] = pd.to_numeric(h1_raw['temp_C'], errors='coerce')
h1 = h1_raw.set_index('datetime').sort_index()[['pH','temp_C']].dropna()
h1_res = h1['pH'].resample('1min').median().dropna()
# Hannah S1: ATC internal — no Nernst
pH_h1 = h1_res.copy()
print(f'Hannah S1: {len(pH_h1)} pts | pH {pH_h1.min():.3f}→{pH_h1.max():.3f} | {pH_h1.index[0].time()}→{pH_h1.index[-1].time()}')

# ── Session 2 — Hannah (PHLOT002) ───────────────────────────────────────
h2_raw = pd.read_csv('data_5_mai/trace_pH_hannah_05052026.csv', parse_dates=['datetime'])
h2 = h2_raw.set_index('datetime').sort_index()[['pH','temp_C']].dropna()
h2_res = h2['pH'].resample('1min').median().dropna()
# Hannah S2: ATC internal — no Nernst
pH_h2 = h2_res.copy()
print(f'Hannah S2: {len(pH_h2)} pts | pH {pH_h2.min():.3f}→{pH_h2.max():.3f} | {pH_h2.index[0].time()}→{pH_h2.index[-1].time()}')

# ── Align to t = 0 (moulage) ────────────────────────────────────────────
def align_to_t0(series, T0):
    """Convert datetime index to hours elapsed since T0."""
    return pd.Series(
        series.values,
        index=(series.index - T0).total_seconds() / 3600,
        name=series.name
    )

pH_h1_t = align_to_t0(pH_h1, SESSIONS['S1']['T0_moulage'])
pH_h2_t = align_to_t0(pH_h2, SESSIONS['S2']['T0_moulage'])
print('\nAligned to t=0 (moulage)')
print(f'S1 t range: {pH_h1_t.index[0]:.2f}h → {pH_h1_t.index[-1]:.2f}h')
print(f'S2 t range: {pH_h2_t.index[0]:.2f}h → {pH_h2_t.index[-1]:.2f}h')


---
## 2. pH Profiles — Cross-Session Comparison (Hannah, aligned to t = 0)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(pH_h1_t.index, pH_h1_t.values,
        color=SESSIONS['S1']['color'], linewidth=2.0,
        label=f"{SESSIONS['S1']['label']} — yield {SESSIONS['S1']['yield_pct']}%")
ax.plot(pH_h2_t.index, pH_h2_t.values,
        color=SESSIONS['S2']['color'], linewidth=2.0,
        label=f"{SESSIONS['S2']['label']} — yield {SESSIONS['S2']['yield_pct']}%")

ax.axvline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.5, label='t = 0 (moulage)')

for th, ls in [(5.3, ':'), (5.0, '--')]:
    ax.axhline(th, color='grey', linewidth=0.8, linestyle=ls, alpha=0.6)
    ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else -0.5,
            th + 0.02, f'pH {th}', fontsize=8, color='grey')

ax.set_xlabel('Time since moulage (h)')
ax.set_ylabel('pH (Hannah — ATC internal)')
ax.set_title('Cross-Session pH Comparison — Hannah probe (no Nernst)', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cmp_01_pH_profiles.png', bbox_inches='tight')
plt.show()


---
## 3. Key Features — Acidification Milestones

In [ ]:
def extract_features(ph_t, label, yield_pct):
    """Extract key quantitative features from a pH-vs-time series (t in hours)."""
    ph = ph_t.dropna()
    # Rate of acidification
    dpH = np.gradient(ph.values, ph.index)
    max_rate_idx = np.argmin(dpH)   # most negative = fastest drop

    def first_below(threshold):
        mask = ph < threshold
        return round(ph.index[mask].min(), 2) if mask.any() else None

    return {
        'Session':          label,
        'Yield (%)':        yield_pct,
        'pH initial':       round(ph.iloc[0], 3),
        'pH final':         round(ph.iloc[-1], 3),
        'ΔpH total':        round(ph.iloc[0] - ph.iloc[-1], 3),
        't → pH 5.5 (h)':   first_below(5.5),
        't → pH 5.3 (h)':   first_below(5.3),
        't → pH 5.0 (h)':   first_below(5.0),
        'Max rate (pH/h)':  round(abs(dpH[max_rate_idx]), 4),
        't max rate (h)':   round(ph.index[max_rate_idx], 2),
    }

rows = [
    extract_features(pH_h1_t, SESSIONS['S1']['label'], SESSIONS['S1']['yield_pct']),
    extract_features(pH_h2_t, SESSIONS['S2']['label'], SESSIONS['S2']['yield_pct']),
]
feat_df = pd.DataFrame(rows).set_index('Session')
feat_df


---
## 4. Logistic Decline Fit — Parameter Comparison

$$\mathrm{pH}(t) = L + \frac{\mathrm{pH}_0 - L}{1 + e^{k(t-t_0)}}$$

- **L** : asymptotic minimum pH (plateau)
- **pH₀** : initial pH at t = 0
- **k** : acidification rate constant (h⁻¹)
- **t₀** : inflection time (h after moulage)


In [ ]:
def logistic_decline(t, L, pH0, k, t0):
    return L + (pH0 - L) / (1 + np.exp(k * (t - t0)))

def fit_logistic_t(ph_t, fw_start_h, fw_end_h):
    """Fit logistic on aligned (hours) series; return params + fit-window index."""
    mask  = (ph_t.index >= fw_start_h) & (ph_t.index <= fw_end_h)
    t_fit = ph_t.index[mask]
    y_fit = ph_t.values[mask]
    if len(t_fit) < 4:
        return None
    p0 = [y_fit.min(), y_fit.max(), 2.0, np.median(t_fit)]
    try:
        popt, pcov = curve_fit(logistic_decline, t_fit, y_fit, p0=p0, maxfev=10000)
        y_pred = logistic_decline(t_fit, *popt)
        ss_res = np.sum((y_fit - y_pred) ** 2)
        ss_tot = np.sum((y_fit - y_fit.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot
        perr = np.sqrt(np.diag(pcov))
        return dict(L=popt[0], pH0=popt[1], k=popt[2], t0=popt[3],
                    r2=r2, t_fit=t_fit, y_pred=y_pred, perr=perr)
    except Exception as e:
        print(f'Fit failed: {e}'); return None

# Convert FW timestamps to hours-since-moulage
def ts_to_h(ts, T0): return (ts - T0).total_seconds() / 3600

fit_h1 = fit_logistic_t(
    pH_h1_t,
    ts_to_h(SESSIONS['S1']['FW_START'], SESSIONS['S1']['T0_moulage']),
    ts_to_h(SESSIONS['S1']['FW_END'],   SESSIONS['S1']['T0_moulage'])
)
fit_h2 = fit_logistic_t(
    pH_h2_t,
    ts_to_h(SESSIONS['S2']['FW_START'], SESSIONS['S2']['T0_moulage']),
    ts_to_h(SESSIONS['S2']['FW_END'],   SESSIONS['S2']['T0_moulage'])
)

for key, fit, sess in [('S1', fit_h1, 'S1'), ('S2', fit_h2, 'S2')]:
    print(f"{SESSIONS[sess]['label']}:")
    print(f"  L={fit['L']:.3f}  pH0={fit['pH0']:.3f}  k={fit['k']:.3f}  t0={fit['t0']:.3f}  R²={fit['r2']:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (sess_key, ph_t, fit) in zip(axes, [
        ('S1', pH_h1_t, fit_h1),
        ('S2', pH_h2_t, fit_h2)]):
    s = SESSIONS[sess_key]
    ax.plot(ph_t.index, ph_t.values, color=s['color'], linewidth=1.4, alpha=0.7,
            label='Hannah (mesure)')
    if fit:
        ax.plot(fit['t_fit'], fit['y_pred'], color='black', linewidth=2.2,
                linestyle='--',
                label=f'Logistic fit (R²={fit["r2"]:.3f})')
        ax.axhline(fit['L'],  color='red',    linestyle=':', linewidth=1,
                   label=f'L = {fit["L"]:.3f}')
        ax.axvline(fit['t0'], color='orange', linestyle=':', linewidth=1,
                   label=f't₀ = {fit["t0"]:.2f} h')
    ax.set_title(s['label'] + f"  —  yield {s['yield_pct']}%", fontweight='bold')
    ax.set_xlabel('Time since moulage (h)')
    ax.set_ylabel('pH')
    ax.legend(fontsize=9)

plt.suptitle('Logistic Decline Fit — Hannah probe (both sessions)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cmp_02_logistic_fit.png', bbox_inches='tight')
plt.show()


---
## 5. Logistic Parameters vs Fresh Yield

> ⚠️ With only 2 sessions, the correlations below are **indicative only** — they show the direction of the relationship but cannot be validated statistically. Additional sessions are needed for significance testing.


In [ ]:
params_logistic = []
for sess_key, fit in [('S1', fit_h1), ('S2', fit_h2)]:
    s = SESSIONS[sess_key]
    params_logistic.append({
        'Session':    s['label'],
        'Yield (%)':  s['yield_pct'],
        'L (pH plateau)':   round(fit['L'],   3),
        'pH₀ (initial)':    round(fit['pH0'], 3),
        'k (rate, h⁻¹)':    round(fit['k'],   3),
        't₀ (inflection h)':round(fit['t0'],  3),
        'R²':               round(fit['r2'],  3),
    })
log_df = pd.DataFrame(params_logistic).set_index('Session')
print(log_df.to_string())
log_df


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
yields = [SESSIONS['S1']['yield_pct'], SESSIONS['S2']['yield_pct']]
colors = [SESSIONS['S1']['color'],     SESSIONS['S2']['color']]
labels = ['S1', 'S2']

param_pairs = [
    ('L (pH plateau)',    [fit_h1['L'],   fit_h2['L']]),
    ('k (rate, h⁻¹)',     [fit_h1['k'],   fit_h2['k']]),
    ('t₀ (inflection h)', [fit_h1['t0'],  fit_h2['t0']]),
    ('pH₀ (initial)',     [fit_h1['pH0'], fit_h2['pH0']]),
]

for ax, (param_name, values) in zip(axes, param_pairs):
    for y, v, c, l in zip(yields, values, colors, labels):
        ax.scatter(v, y, color=c, s=120, zorder=3, label=l)
        ax.annotate(l, (v, y), textcoords='offset points', xytext=(5, 3), fontsize=9)
    ax.set_xlabel(param_name)
    ax.set_ylabel('Fresh yield (%)')
    ax.set_title(f'Yield vs {param_name}', fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Logistic parameters vs Fresh Yield (Hannah probe)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cmp_03_logistic_vs_yield.png', bbox_inches='tight')
plt.show()


---
## 6. Luedeking-Piret ODE Model — Cross-Session Comparison

$$\frac{dX}{dt} = \mu \cdot X \cdot (1 - X) \qquad\frac{dA}{dt} = \alpha \frac{dX}{dt} + \beta X$$

- **μ** : specific growth rate (h⁻¹)
- **α** : growth-associated acid production coefficient
- **β** : non-growth-associated coefficient
- **A(t)** = pH₀ − pH(t) (lactic acid proxy)


In [ ]:
def lp_ode(t, y, mu, alpha, beta):
    X, A = y
    dXdt = mu * X * (1 - X)
    dAdt = alpha * dXdt + beta * X
    return [dXdt, dAdt]

def fit_lp_ode_t(ph_t, fw_start_h, fw_end_h):
    ph = ph_t[(ph_t.index >= fw_start_h) & (ph_t.index <= fw_end_h)].dropna()
    ph = ph.rolling(window=5, center=True, min_periods=1).median()
    t     = ph.index.values.astype(float)
    A_obs = np.clip(ph.values[0] - ph.values, 0, None)
    X0, A0 = 0.01, 0.0

    def residuals(params):
        mu, alpha, beta = params
        if mu <= 0 or alpha < 0 or beta < 0:
            return 1e10
        sol = solve_ivp(lp_ode, [t[0], t[-1]], [X0, A0],
                        args=(mu, alpha, beta),
                        t_eval=t, method='RK45', rtol=1e-6, atol=1e-8)
        if not sol.success or sol.y.shape[1] != len(t):
            return 1e10
        return np.sum((sol.y[1] - A_obs) ** 2)

    res = minimize(residuals, x0=[2.0, 0.3, 0.05], method='Nelder-Mead',
                   options={'xatol': 1e-6, 'fatol': 1e-8, 'maxiter': 8000})
    mu_f, alpha_f, beta_f = res.x

    sol = solve_ivp(lp_ode, [t[0], t[-1]], [X0, A0],
                    args=(mu_f, alpha_f, beta_f),
                    t_eval=t, method='RK45', rtol=1e-6, atol=1e-8)
    A_sim = sol.y[1]; X_sim = sol.y[0]
    ss_res = np.sum((A_sim - A_obs) ** 2)
    ss_tot = np.sum((A_obs - A_obs.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return dict(t=t, A_obs=A_obs, A_sim=A_sim, X_sim=X_sim,
                mu=mu_f, alpha=alpha_f, beta=beta_f, r2=r2)

lp_h1 = fit_lp_ode_t(
    pH_h1_t,
    ts_to_h(SESSIONS['S1']['FW_START'], SESSIONS['S1']['T0_moulage']),
    ts_to_h(SESSIONS['S1']['FW_END'],   SESSIONS['S1']['T0_moulage'])
)
lp_h2 = fit_lp_ode_t(
    pH_h2_t,
    ts_to_h(SESSIONS['S2']['FW_START'], SESSIONS['S2']['T0_moulage']),
    ts_to_h(SESSIONS['S2']['FW_END'],   SESSIONS['S2']['T0_moulage'])
)

for key, lp, sess in [('S1', lp_h1, 'S1'), ('S2', lp_h2, 'S2')]:
    print(f"{SESSIONS[sess]['label']}:")
    print(f"  μ={lp['mu']:.3f}  α={lp['alpha']:.4f}  β={lp['beta']:.4f}  R²={lp['r2']:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (sess_key, lp) in zip(axes, [('S1', lp_h1), ('S2', lp_h2)]):
    s = SESSIONS[sess_key]
    ax.plot(lp['t'], lp['A_obs'], color='lightgrey', linewidth=1.4, label='A(t) observé')
    ax.plot(lp['t'], lp['A_sim'], color=s['color'],  linewidth=2.2,
            linestyle='--', label=f'ODE fit (R²={lp["r2"]:.3f})')

    ax2 = ax.twinx()
    ax2.plot(lp['t'], lp['X_sim'], color='darkorange', linewidth=1.2,
             linestyle=':', alpha=0.7, label='X(t) simulé')
    ax2.set_ylabel('X (biomasse norm.)', color='darkorange', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='darkorange')
    ax2.set_ylim(0, 1.2)

    ax.text(0.04, 0.94,
            f'μ = {lp["mu"]:.3f}\nα = {lp["alpha"]:.4f}\nβ = {lp["beta"]:.4f}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_title(s['label'] + f"  —  yield {s['yield_pct']}%", fontweight='bold')
    ax.set_xlabel('Time since moulage (h)')
    ax.set_ylabel('A(t) = pH₀ − pH(t)')
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Luedeking-Piret ODE — Both Sessions (Hannah probe)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cmp_04_lp_ode.png', bbox_inches='tight')
plt.show()


---
## 7. LP ODE Parameters vs Fresh Yield

In [ ]:
params_lp = []
for sess_key, lp in [('S1', lp_h1), ('S2', lp_h2)]:
    s = SESSIONS[sess_key]
    params_lp.append({
        'Session':          s['label'],
        'Yield (%)':        s['yield_pct'],
        'μ (growth rate)':  round(lp['mu'],    3),
        'α (growth-assoc)': round(lp['alpha'], 4),
        'β (non-growth)':   round(lp['beta'],  4),
        'α/β':              round(lp['alpha'] / lp['beta'], 2) if lp['beta'] > 1e-6 else '∞',
        'R²':               round(lp['r2'],    3),
    })
lp_df = pd.DataFrame(params_lp).set_index('Session')
print(lp_df.to_string())
lp_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
yields = [SESSIONS['S1']['yield_pct'], SESSIONS['S2']['yield_pct']]
colors = [SESSIONS['S1']['color'],     SESSIONS['S2']['color']]

param_pairs_lp = [
    ('μ (growth rate, h⁻¹)', [lp_h1['mu'],    lp_h2['mu']]),
    ('α (growth-assoc.)',    [lp_h1['alpha'],  lp_h2['alpha']]),
    ('β (non-growth-assoc.)',[lp_h1['beta'],   lp_h2['beta']]),
]

for ax, (param_name, values) in zip(axes, param_pairs_lp):
    for y, v, c, l in zip(yields, values, colors, ['S1','S2']):
        ax.scatter(v, y, color=c, s=140, zorder=3, label=l)
        ax.annotate(l, (v, y), textcoords='offset points', xytext=(5, 3), fontsize=9)
    ax.set_xlabel(param_name)
    ax.set_ylabel('Fresh yield (%)')
    ax.set_title(f'Yield vs {param_name}', fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('LP ODE parameters vs Fresh Yield (Hannah probe)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/cmp_05_lp_vs_yield.png', bbox_inches='tight')
plt.show()


---
## 8. Summary — All Parameters

In [ ]:
summary = []
for sess_key, fit, lp in [('S1', fit_h1, lp_h1), ('S2', fit_h2, lp_h2)]:
    s = SESSIONS[sess_key]
    summary.append({
        'Session':           s['label'],
        'Cultures':          s['cultures'],
        'Yield (%)':         s['yield_pct'],
        '── Logistic ──':    '',
        'L':                 round(fit['L'],   3),
        'k (h⁻¹)':           round(fit['k'],   3),
        't₀ (h)':            round(fit['t0'],  3),
        'R² logistic':       round(fit['r2'],  3),
        '── LP ODE ──':      '',
        'μ (h⁻¹)':           round(lp['mu'],    3),
        'α':                 round(lp['alpha'], 4),
        'β':                 round(lp['beta'],  4),
        'R² ODE':            round(lp['r2'],    3),
    })
summary_df = pd.DataFrame(summary).set_index('Session')
summary_df.T


---
## 9. Observations & Interpretation

### Logistic fit (L, k, t₀)
| Parameter | Scientific meaning | Link to yield |
|-----------|-------------------|---------------|
| **L** (plateau) | Final pH reached — lower = more acidic curd | Lower L → stronger acidification → potentially lower yield (more whey expelled) |
| **k** (rate) | Speed of acidification | Faster acidification may reduce gel time and affect texture |
| **t₀** (inflection) | Time of maximum acidification speed | Earlier t₀ → cultures more active earlier |

### LP ODE (μ, α, β)
| Parameter | Scientific meaning |
|-----------|-------------------|
| **μ** | Bacterial growth rate — linked to culture type and ratio |
| **α** | Growth-associated acid production (exponential phase) |
| **β** | Non-growth-associated (stationary phase — linked to residual activity) |

> **Culture ratio hypothesis:** Session 1 used MFR38-dominant (3.45:1) giving higher yield (12.17%). Session 2 used MFR32-dominant (1:2) giving lower yield (11.80%). If μ, α, β differ significantly between sessions, this would support the hypothesis that culture ratio is a key driver of both acidification kinetics and fresh yield.

> **Next step:** Add a 3rd session with a different culture ratio to confirm or reject this hypothesis.
